AULA 07, PARTE D: LSA E BUSCA HÍBRIDA (RRF) NO CORPUS DO GRUPO



1. O MOTOR, O CORPUS E O ÍNDICE (IGUAL À AULA 06)

In [ ]:
REPO <- "https://raw.githubusercontent.com/luisfeliperuas18-tech/PI3-LUISFELIPERUAS-LORENZOLOUZADA-ARTHURDAVINO/main/"   # o repositorio do grupo
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")   # funcoes do curso, agora com lsa, ranking_lsa e rrf
cfg <- cfg_aula()                                    # decisoes do curso (limiar = 2)
cfg$acentos <- "translit"                            # a decisao do grupo na Aula 06: tira acentos
cp <- read.csv(paste0(REPO, "estrutura/codigo/corpus/corpus_baixada_santista.csv"),   # o corpus do grupo
               stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")                   #   UTF-8 com BOM
docs <- setNames(cp$texto, cp$id)                    # vetor nomeado id -> texto
ix <- montar(docs, cfg)                              # o indice, com a limpeza do grupo
dim(ix$w)                                            # termos x documentos: deve dar 4757 x 220

2. JULGAMENTOS E CONSULTAS (IGUAL À AULA 06)

In [ ]:
rs <- read.csv("qrels_consolidado_resolvidos.csv", stringsAsFactors = FALSE)   # consulta, documento, grau
qs <- unique(rs$consulta)                                                      # as consultas julgadas
qrels <- lapply(qs, function(q) setNames(rs$grau[rs$consulta == q], rs$documento[rs$consulta == q]))   # graus por consulta
names(qrels) <- qs                                                             # q01 ... q05
nec <- read.csv(paste0(REPO, "estrutura/codigo/to%20delete/5-entrega-necessidades.csv"),   # as 5 necessidades
                stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)                        # q01 -> texto digitado
sapply(qrels, function(g) sum(g >= cfg$limiar))                                # relevantes por consulta: 2 2 2 1 2

3. A LSA COM K DIMENSÕES (TAREFA 1)

In [ ]:
Ndoc <- ncol(ix$w)                                   # numero de documentos: 220
m20 <- lsa(ix, 20)                                   # SVD do TF-IDF, guardando 20 dimensoes
dim(m20$U); length(m20$S); dim(m20$D)                # U: termos x k; S: k valores; D: documentos x k
round(m20$S[1:5], 2)                                 # os primeiros valores singulares

4. TOP 5 DO BM25 CONTRA O DA LSA NA q01, LENDO OS TEXTOS (TAREFA 1)

In [ ]:
q <- "q01"                                                            # a consulta de trabalho: "fundacao de Santos"
b   <- ranking_bm25(consultas[[q]], ix, cfg)                          # ranking BM25
l20 <- ranking_lsa(consultas[[q]], ix, cfg, m20)                      # ranking LSA com k = 20
data.frame(bm25 = names(b)[1:5], grau_bm25 = unname(qrels[[q]][names(b)[1:5]]),         # top 5 e graus (NA = nao julgado)
           lsa20 = names(l20)[1:5], grau_lsa = unname(qrels[[q]][names(l20)[1:5]]))
match(names(qrels[[q]])[qrels[[q]] >= cfg$limiar], names(b))          # posicao dos relevantes no BM25
match(names(qrels[[q]])[qrels[[q]] >= cfg$limiar], names(l20))        # ...e na LSA
for (i in names(l20)[1:5]) cat("\n[", i, "| grau:", qrels[[q]][i], "]\n", substr(docs[[i]], 1, 250), "\n")   # leiam os textos da LSA
for (i in names(b)[1:5])   cat("\n[", i, "| grau:", qrels[[q]][i], "]\n", substr(docs[[i]], 1, 250), "\n")   # e os do BM25

5. MAP NO DEV: BM25, LSA COM VÁRIOS K E RRF (TAREFA 2)

In [ ]:
dev   <- c("q01", "q02", "q03")                      # aqui se escolhe modelo e k
teste <- c("q04", "q05")                             # so no fim, uma vez
ks <- c(10, 20, 50, Ndoc)                            # os k do slide: 10, 20, 50 e ate o numero de documentos
mods <- setNames(lapply(ks, function(k) lsa(ix, k)), paste0("k", ks))   # uma SVD por k
rank_de <- function(nome, txt, c = 60) {             # o ranking de um candidato, pelo nome
  if (nome == "bm25") return(ranking_bm25(txt, ix, cfg))
  k <- sub("^[a-z]+_", "", nome)                     # "lsa_k20" -> "k20"
  if (startsWith(nome, "lsa_")) return(ranking_lsa(txt, ix, cfg, mods[[k]]))
  rrf(ranking_bm25(txt, ix, cfg), ranking_lsa(txt, ix, cfg, mods[[k]]), c = c)   # hib_k20: BM25 + LSA k = 20
}
cands <- c("bm25", paste0("lsa_k", ks), paste0("hib_k", ks))             # do mais simples ao mais complexo
ap_de <- function(nome, q) metricas(rank_de(nome, consultas[[q]]), qrels[[q]], cfg$limiar)[["AP"]]   # AP de um candidato numa consulta
tab_dev <- t(sapply(cands, function(n) sapply(dev, function(q) ap_de(n, q))))   # candidato x consulta
colnames(tab_dev) <- dev
round(cbind(tab_dev, MAP = rowMeans(tab_dev)), 3)    # o MAP de cada candidato

6. A DECISÃO NO DEV (REGRA FIXADA ANTES DE OLHAR O TESTE)

In [ ]:
map_dev <- rowMeans(tab_dev)                                   # MAP de cada candidato
perto <- names(map_dev)[map_dev >= max(map_dev) - 0.005]       # os que empatam com o melhor (ate 0.005)
escolhido <- perto[1]                                          # empate: o mais simples (ordem de cands)
escolhido; round(map_dev[escolhido], 3)

7. O TESTE, UMA ÚNICA VEZ (q04 E q05)

In [ ]:
tab_teste <- sapply(teste, function(q) c(bm25 = ap_de("bm25", q), escolhido = ap_de(escolhido, q)))   # AP por consulta
round(tab_teste, 3); round(rowMeans(tab_teste), 3)             # por consulta e o MAP

8. REGISTRANDO A DECISÃO NO config.R (TAREFA 3)

In [ ]:
# Os valores abaixo vem da CELULA 6. Se o escolhido mudar, mude aqui tambem.
writeLines(c(
  "cfg <- list(",
  "  limpar    = TRUE,                           # limpeza da Aula 03",
  "  acentos   = \"translit\",                     # tira acentos (decidido na Aula 06)",
  "  stopwords = character(0),                   # sem lista de stopwords",
  "  k1 = 1.2, b = 0.75,                         # BM25",
  "  limiar    = 2,                              # grau 2 conta como relevante",
  "  # --- Aula 06: realimentacao ---",
  "  prf_k   = 0,                                # 0 = sem pseudo-realimentacao",
  "  rocchio = c(a = 1, b = 0.75, g = 0.15),     # pesos do Rocchio",
  "  # --- Aula 07: LSA e hibrido ---",
  "  modelo  = \"lsa\",                            # decisao no dev: \"bm25\", \"lsa\" ou \"hibrido\"",
  "  lsa_k   = 220,                              # k escolhido no dev (220 = numero de documentos)",
  "  rrf_c   = 60                                # constante do RRF (padrao de Cormack et al.; nao foi ajustada)",
  ")"), "config.R")                               # grava o arquivo
source("config.R")                                 # recarrega o cfg
cfg$modelo; cfg$lsa_k; cfg$rrf_c                   # confiram com a decisao

ranking_motor <- function(txt, ix, cfg, m = NULL) {                            # o ranking do motor, pela decisao do config
  if (cfg$modelo == "bm25") return(ranking_bm25(txt, ix, cfg))               # lexical
  if (is.null(m)) m <- lsa(ix, cfg$lsa_k)                                    # a LSA com o k do config
  if (cfg$modelo == "lsa") return(ranking_lsa(txt, ix, cfg, m))              # so LSA
  rrf(ranking_bm25(txt, ix, cfg), ranking_lsa(txt, ix, cfg, m), c = cfg$rrf_c)   # hibrido por RRF
}
round(head(ranking_motor(consultas[["q01"]], ix, cfg), 5), 3)                  # o top 5 que o motor daria na q01

9. FICHA E CONSOLIDADO

In [ ]:
list.files()                                       # tem que mostrar config.R, o qrels e os .md
# Depois de salvar o consolidado (aula07_parteD_consolidado.md) e a ficha (00_FICHA_PROJETO.md) na pasta:
# anexar_estado("00_FICHA_PROJETO.md")
# anexar_estado("aula07_parteD_consolidado.md")